In [0]:
#import statements
from pyspark.sql import Row

In [0]:
new_rows = [
  Row(claim_id=101, policy_id=1, claim_date='2024-01-05', amount=5000.0, status='CLOSED'),  # update
  Row(claim_id=102, policy_id=2, claim_date='2024-01-12', amount=3000.0, status='CLOSED'),  # update
  Row(claim_id=111, policy_id=3, claim_date='2024-04-01', amount=7500.0, status='OPEN'),   # new
  Row(claim_id=112, policy_id=1, claim_date='2024-04-05', amount=2000.0, status='PENDING'),# new
  Row(claim_id=113, policy_id=4, claim_date='2024-04-10', amount=9000.0, status='OPEN'),   # new
]

In [0]:
new_claims_df = spark.createDataFrame(new_rows)

In [0]:
from pyspark.sql.functions import col, upper, to_date

new_claims_df = new_claims_df.select(
    col("claim_id").cast("int"),
    col("policy_id").cast("int"),
    to_date(col("claim_date")).alias("claim_date"),
    col("amount").cast("double").alias("claim_amount"),
    upper(col("status")).alias("status")
)


In [0]:
new_claims_df.columns

['claim_id', 'policy_id', 'claim_date', 'claim_amount', 'status']

In [0]:
new_claims_df.createOrReplaceTempView("new_claims")

In [0]:
%sql
MERGE INTO insurance_db.silver.claims AS target
USING new_claims AS source
ON target.claim_id = source.claim_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *

In [0]:
spark.table("insurance_db.silver.claims").show()


+--------+---------+------------+----------+--------+
|claim_id|policy_id|claim_amount|claim_date|  status|
+--------+---------+------------+----------+--------+
|     105|        5|       700.0|2024-06-01|APPROVED|
|     109|        9|       350.0|2024-10-01| pending|
|     104|        4|       150.0|2024-05-01|approved|
|     108|        8|       600.0|2024-09-01|  denied|
|     103|        3|       300.0|2024-04-01|  denied|
|     101|        1|      5000.0|2024-01-05|  CLOSED|
|     102|        2|      3000.0|2024-01-12|  CLOSED|
|     111|        3|      7500.0|2024-04-01|    OPEN|
|     112|        1|      2000.0|2024-04-05| PENDING|
|     113|        4|      9000.0|2024-04-10|    OPEN|
+--------+---------+------------+----------+--------+

